In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## Load Household Power Consumption Dataset

In this section, we load the **Household Power Consumption** dataset using `pandas`.

The dataset contains household electricity consumption measurements recorded at one-minute intervals.

When loading the dataset:
- The columns are separated using `;`.
- The `?` character is treated as a missing value.

In [2]:
power_ds = pd.read_csv("/mnt/c/Machine Learning & Deep Learning/datasets/household_power_consumption.txt", 
                       sep= ';', na_values= ['?'])
power_ds.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Dataset Information

In this step, we use the `info()` method to examine the overall structure of the dataset, including the number of observations, columns, and data types.

The dataset contains **2,075,259 records** and **9 columns**.

The columns can be categorized as follows:
- `Date` and `Time` are stored as strings.
- The remaining 7 columns contain numerical measurements stored as `float64`.

In [3]:
power_ds.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 142.5 MB


## Missing Values

In this step, we check the percentage of missing values in each column.

We calculate the mean of `isna()` for each column and multiply it by 100 to obtain the percentage of missing values.

This analysis helps us understand the quality of the dataset before performing **data preprocessing** and **modeling**.

In [4]:
power_ds.isna().mean() * 100

Date                     0.000000
Time                     0.000000
Global_active_power      1.251844
Global_reactive_power    1.251844
Voltage                  1.251844
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           1.251844
dtype: float64

### Value Frequency Analysis

The `value_counts()` method is used to examine the frequency of each unique value in the `Global_active_power` column.

This helps identify the most frequent power consumption values and provides an overview of how the data is distributed.

In [5]:
power_ds['Global_active_power'].value_counts()

Global_active_power
0.218    9565
0.216    9363
0.322    9350
0.324    9304
0.326    9187
         ... 
9.724       1
9.656       1
9.630       1
9.648       1
8.600       1
Name: count, Length: 4186, dtype: int64

### Outlier Detection Using the IQR Method

The **Interquartile Range (IQR)** method is used to identify potential outliers in the `Global_active_power` column.

- **Q1 (25th percentile):** Represents the lower quartile.
- **Q3 (75th percentile):** Represents the upper quartile.
- **IQR:** Calculated as `Q3 - Q1`.
- **Lower Bound:** `Q1 - 1.5 × IQR`
- **Upper Bound:** `Q3 + 1.5 × IQR`

Any value below the lower bound or above the upper bound is considered a potential outlier.

In this dataset, **94,907 observations** are identified as potential outliers using this method.

In [16]:
q1 = power_ds['Global_active_power'].quantile(0.25)
q3 = power_ds['Global_active_power'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = power_ds[(power_ds['Global_active_power'] < lower_bound) | (power_ds['Global_active_power'] > upper_bound)]
print(len(outliers))
print(outliers['Global_active_power'])

94907
0          4.216
1          5.360
2          5.374
3          5.388
4          3.666
           ...  
2074573    3.632
2074574    3.616
2074575    3.632
2074576    3.552
2074577    3.552
Name: Global_active_power, Length: 94907, dtype: float64


### Descriptive Statistics

The `describe()` method is used to obtain a statistical summary of the `Global_active_power` column.

It provides key statistics such as the count, mean, standard deviation, minimum, quartiles, and maximum values. This helps us understand the overall distribution and variability of the power consumption data.

In [19]:
power_ds['Global_active_power'].describe()

count    2.049280e+06
mean     1.091615e+00
std      1.057294e+00
min      7.600000e-02
25%      3.080000e-01
50%      6.020000e-01
75%      1.528000e+00
max      1.112200e+01
Name: Global_active_power, dtype: float64

### Skewness Analysis

The skewness of numerical features is analyzed to determine an appropriate method for handling missing values.

To avoid repeating the same conditional logic for each feature, a reusable function was defined. The function calculates the skewness of a given column and automatically recommends either **mean** or **median** imputation based on the distribution's skewness.

For approximately symmetric distributions, the function recommends **mean imputation**. For moderately or highly skewed distributions, it recommends **median imputation**, as the median is more robust to extreme values and outliers.

For `Global_active_power`, the calculated skewness is approximately **1.81**, indicating a positively skewed distribution. Therefore, the function recommends **median imputation**.

In [28]:
def checked_how_to_impute_nan_values (column_name):
    check_skew = column_name.skew()
    print(check_skew)
    if check_skew >= 1 or check_skew <= -1:
        print("Use Median for imputation")
    elif -0.5 <= check_skew <= 0.5:
        print("Use Mean for imputation")
    elif (-1 < check_skew < -0.5) or (0.5 < check_skew < 1):
        print("Use Median for imputation")

In [29]:
checked_how_to_impute_nan_values(power_ds['Global_active_power'])

1.8065219476816126
Use Median for imputation


### Missing Value Imputation

Since `Global_active_power` is positively skewed and contains potential outliers, the **median** is used to replace its missing values.

After imputation, the percentage of missing values in each column is calculated again to verify that the missing values in `Global_active_power` have been successfully handled.

In [20]:
power_ds['Global_active_power'] = power_ds['Global_active_power'].fillna(power_ds['Global_active_power'].median())

In [21]:
power_ds.isna().mean() * 100

Date                     0.000000
Time                     0.000000
Global_active_power      0.000000
Global_reactive_power    1.251844
Voltage                  1.251844
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           1.251844
dtype: float64

### Descriptive Statistics — `Voltage`

The `describe()` method is used to examine the statistical characteristics of the `Voltage` feature.

The results show that the voltage is centered around **240.84 V**, with a median of approximately **241.01 V** and a standard deviation of **3.24 V**.

The middle 50% of the observations are between **238.99 V** and **242.89 V**, while the minimum and maximum values are **223.20 V** and **254.15 V**, respectively.

Overall, the relatively small standard deviation indicates that the voltage values are concentrated within a narrow range.

In [22]:
power_ds['Voltage'].describe()

count    2.049280e+06
mean     2.408399e+02
std      3.239987e+00
min      2.232000e+02
25%      2.389900e+02
50%      2.410100e+02
75%      2.428900e+02
max      2.541500e+02
Name: Voltage, dtype: float64

### Statistical Outlier Analysis — `Voltage`

The **Interquartile Range (IQR)** method is used to identify potential statistical outliers in the `Voltage` feature.

The analysis identifies **51,067 observations** as potential outliers according to the 1.5 × IQR rule.

However, these observations are not automatically considered incorrect or invalid. The detected values are still within a plausible voltage range, with values such as **232–233 V** and a maximum of **254.15 V**.

Therefore, the statistical outliers are **kept in the dataset** rather than removed. The IQR method is used here to identify unusual observations, not to automatically classify them as erroneous measurements.

For this time-series dataset, sudden changes relative to neighboring time points may provide a more meaningful way to investigate abnormal voltage readings in later stages.

In [23]:
q1 = power_ds['Voltage'].quantile(0.25)
q3 = power_ds['Voltage'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = power_ds[(power_ds['Voltage'] < lower_bound) | (power_ds['Voltage'] > upper_bound)]
print(len(outliers))
print(outliers['Voltage'])

51067
10         232.86
11         232.78
12         232.99
13         232.91
20         232.69
            ...  
2072414    232.71
2073685    232.96
2073686    232.98
2073689    233.10
2074342    249.01
Name: Voltage, Length: 51067, dtype: float64


### Missing Value Imputation — `Voltage`

The skewness analysis for the `Voltage` feature returned approximately **-0.33**, indicating that the distribution is relatively close to symmetric.

Based on the defined imputation function, **mean imputation** is selected for this feature.

The missing values in `Voltage` are therefore replaced with the mean value of the column using `fillna()`.

In [30]:
checked_how_to_impute_nan_values(power_ds['Voltage'])

-0.32666497568777597
Use Mean for imputation


In [31]:
power_ds['Voltage'] = power_ds['Voltage'].fillna(power_ds['Voltage'].mean())

### Missing Value Verification

After imputing the missing values in `Voltage`, the percentage of missing values is calculated again for all columns.

The results confirm that the missing values in `Voltage` have been successfully handled, reducing its missing-value percentage to **0%**.

The remaining missing values in other numerical features will be handled in the next preprocessing steps using the same distribution-based approach.

In [32]:
power_ds.isna().mean() * 100

Date                     0.000000
Time                     0.000000
Global_active_power      0.000000
Global_reactive_power    1.251844
Voltage                  0.000000
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           1.251844
dtype: float64